In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import torch
import os
import gc # Garbage Collector
from tqdm.auto import tqdm

print("--- SCRIPT XÂY DỰNG TẤT CẢ INDEX (OFFLINE) ---")

# --- 1. Cấu hình ---

# Định nghĩa các mô hình bạn muốn tạo index
models_to_build = {
    "Model A (Baseline)": "paraphrase-multilingual-mpnet-base-v2",
    "Model B (Our Champion)": "./models/triplet-finetuned-model-v1/",
    "Model C (MiniLM-tuned)": "./models/minilm-finetuned-v1/",
    "Model D (BERT-tuned)": "./models/bert-base-multilingual-cased-finetuned-v1/",
    "Model E (distiluse-tuned)": "./models/distiluse-base-multilingual-cased-v1-finetuned-v1/"
}
# (Lưu ý: Tên key nên đơn giản, không dấu, không cách, để dùng làm tên file)

# Đường dẫn file dữ liệu (corpus)
CORPUS_FILE = 'corpus_with_id.csv'
# Nơi lưu các file index
OUTPUT_DIR = './artifacts/' 
os.makedirs(OUTPUT_DIR, exist_ok=True) # Tạo thư mục nếu chưa có

# --- 2. Kiểm tra GPU ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"--- Đang chạy trên: {device} ---")

# --- 3. Tải Dữ liệu Corpus (Chỉ 1 lần) ---
print(f"Đang tải dữ liệu từ: {CORPUS_FILE}...")
if not os.path.exists(CORPUS_FILE):
    raise FileNotFoundError(f"Không tìm thấy file corpus: {CORPUS_FILE}")

corpus_df = pd.read_csv(CORPUS_FILE)
if 'Question' not in corpus_df.columns:
    raise ValueError("File corpus phải có cột 'Question'.")
documents = corpus_df['Question'].tolist()
print(f"Đã tải {len(documents)} tài liệu.")


# --- 4. Vòng lặp Xây dựng Index ---
print("\n--- BẮT ĐẦU VÒNG LẶP XÂY DỰNG INDEX ---")

for model_name, model_path in models_to_build.items():
    print(f"\n--- Đang xử lý: {model_name} ---")
    print(f"Đường dẫn: {model_path}")
    
    try:
        # 4a. Tải Model
        print("Đang tải model...")
        model = SentenceTransformer(model_path, device=device)
        
        # 4b. Mã hóa (Embedding)
        print(f"Đang mã hóa {len(documents)} tài liệu...")
        doc_embeddings = model.encode(
            documents,
            convert_to_tensor=True,
            show_progress_bar=True,
            batch_size=128 # Tối ưu cho GPU (3050 hoặc T4/V100)
        )
        doc_embeddings_np = doc_embeddings.cpu().numpy()

        # 4c. Xây dựng FAISS Index
        print("Đang xây dựng FAISS Index...")
        dimension = doc_embeddings_np.shape[1]
        faiss.normalize_L2(doc_embeddings_np) # Chuẩn hóa L2 cho Cosine Similarity
        index = faiss.IndexFlatIP(dimension) # IndexFlatIP ~ Cosine Similarity
        index.add(doc_embeddings_np)
        
        # 4d. Lưu Index ra File
        output_index_file = os.path.join(OUTPUT_DIR, f"index_{model_name}.faiss")
        print(f"Đang lưu Index ra file: {output_index_file}...")
        faiss.write_index(index, output_index_file)
        
        print(f"--- Hoàn tất {model_name} ---")

    except Exception as e:
        print(f"LỖI khi xử lý {model_name}: {e}")
        
    finally:
        # Giải phóng bộ nhớ VRAM của GPU trước khi tải model tiếp theo
        del model, doc_embeddings, doc_embeddings_np, index
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()

print("\n--- TOÀN BỘ QUÁ TRÌNH XÂY DỰNG INDEX ĐÃ HOÀN TẤT ---")